# Generacion de ontologia OWL en Turtle con RDFLib

Este cuaderno construye programaticamente una ontologia del dominio laboral colombiano usando RDF, RDFS y OWL con RDFLib.

## Objetivo
- Generar la ontologia en memoria.
- Aplicar clases, jerarquias, propiedades, restricciones e individuos.
- Exportar el resultado en formato Turtle.
- Verificar que se cumplen los minimos solicitados en PARTE C (estructura OWL).

## Requisitos de ejecucion

Si RDFLib no esta instalado, ejecuta la siguiente celda una sola vez.

In [1]:
%pip install rdflib

Note: you may need to restart the kernel to use updated packages.


## 1) Importaciones, namespaces y utilidades

En esta seccion:
- Definimos prefijos estandar (RDF, RDFS, OWL, XSD).
- Creamos un namespace base para la ontologia.
- Definimos funciones auxiliares para evitar repeticion de codigo al agregar clases y propiedades.

In [7]:
from pathlib import Path
from rdflib import BNode, Graph, Literal, Namespace, RDF, RDFS, OWL, XSD
from rdflib.collection import Collection

g = Graph()
EX = Namespace("http://example.org/ontologia-laboral#")

g.bind("ex", EX)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("owl", OWL)
g.bind("xsd", XSD)

def add_class(uri, label_es):
    g.add((uri, RDF.type, OWL.Class))
    g.add((uri, RDFS.label, Literal(label_es, lang="es")))

def add_datatype_property(uri, domain, range_):
    g.add((uri, RDF.type, OWL.DatatypeProperty))
    g.add((uri, RDFS.domain, domain))
    g.add((uri, RDFS.range, range_))

def add_object_property(uri, domain, range_):
    g.add((uri, RDF.type, OWL.ObjectProperty))
    g.add((uri, RDFS.domain, domain))
    g.add((uri, RDFS.range, range_))

## 2) Metadatos de la ontologia

In [3]:
ont = EX.OntologiaLaboral
g.add((ont, RDF.type, OWL.Ontology))
g.add((ont, RDFS.label, Literal("Ontologia OWL del Dominio Laboral Colombiano", lang="es")))
g.add((ont, RDFS.comment, Literal("Ontologia para Knowledge Graph RAG en normativa laboral, construida para la PARTE C de Practica2.", lang="es")))

<Graph identifier=N6af75c0dd1dd49298ae407efd6c92c7e (<class 'rdflib.graph.Graph'>)>

## 3) Clases, jerarquias y disyunciones

In [4]:
classes = {
    "NormaJuridica": "Norma Juridica",
    "LeyLaboral": "Ley Laboral",
    "ActorLaboral": "Actor Laboral",
    "Trabajador": "Trabajador",
    "Empleador": "Empleador",
    "ContratoLaboral": "Contrato Laboral",
    "DerechoLaboral": "Derecho Laboral",
    "ObligacionLaboral": "Obligacion Laboral",
    "SancionLaboral": "Sancion Laboral",
    "AutoridadLaboral": "Autoridad Laboral",
}
for cname, label in classes.items():
    add_class(EX[cname], label)

g.add((EX.LeyLaboral, RDFS.subClassOf, EX.NormaJuridica))
g.add((EX.Trabajador, RDFS.subClassOf, EX.ActorLaboral))
g.add((EX.Empleador, RDFS.subClassOf, EX.ActorLaboral))
g.add((EX.AutoridadLaboral, RDFS.subClassOf, EX.ActorLaboral))

g.add((EX.Trabajador, OWL.disjointWith, EX.Empleador))
g.add((EX.DerechoLaboral, OWL.disjointWith, EX.ObligacionLaboral))

<Graph identifier=N6af75c0dd1dd49298ae407efd6c92c7e (<class 'rdflib.graph.Graph'>)>

## 4) Propiedades e inversas

In [5]:
add_datatype_property(EX.tieneNombre, OWL.Thing, XSD.string)
add_datatype_property(EX.tieneDescripcion, OWL.Thing, XSD.string)
add_datatype_property(EX.tieneAnioPublicacion, EX.NormaJuridica, XSD.gYear)
add_datatype_property(EX.tieneNumeroNorma, EX.NormaJuridica, XSD.string)
add_datatype_property(EX.salarioMensual, EX.ContratoLaboral, XSD.decimal)
add_datatype_property(EX.fechaInicioContrato, EX.ContratoLaboral, XSD.date)
add_datatype_property(EX.horasSemanales, EX.ContratoLaboral, XSD.integer)

add_object_property(EX.tieneRelacionConActor, EX.ContratoLaboral, EX.ActorLaboral)
add_object_property(EX.tieneTrabajador, EX.ContratoLaboral, EX.Trabajador)
add_object_property(EX.participaEnContrato, EX.Trabajador, EX.ContratoLaboral)
add_object_property(EX.tieneEmpleador, EX.ContratoLaboral, EX.Empleador)
add_object_property(EX.otorgaDerecho, EX.NormaJuridica, EX.DerechoLaboral)
add_object_property(EX.esOtorgadoPor, EX.DerechoLaboral, EX.NormaJuridica)
add_object_property(EX.imponeObligacion, EX.NormaJuridica, EX.ObligacionLaboral)
add_object_property(EX.estableceSancion, EX.NormaJuridica, EX.SancionLaboral)
add_object_property(EX.afectaAActor, EX.SancionLaboral, EX.ActorLaboral)
add_object_property(EX.imponeSancionA, EX.AutoridadLaboral, EX.ActorLaboral)
add_object_property(EX.supervisaCumplimiento, EX.AutoridadLaboral, EX.ObligacionLaboral)
add_object_property(EX.beneficiaA, EX.DerechoLaboral, EX.ActorLaboral)

g.add((EX.tieneTrabajador, RDFS.subPropertyOf, EX.tieneRelacionConActor))
g.add((EX.tieneEmpleador, RDFS.subPropertyOf, EX.tieneRelacionConActor))
g.add((EX.imponeSancionA, RDFS.subPropertyOf, EX.afectaAActor))

g.add((EX.tieneTrabajador, OWL.inverseOf, EX.participaEnContrato))
g.add((EX.otorgaDerecho, OWL.inverseOf, EX.esOtorgadoPor))

<Graph identifier=N6af75c0dd1dd49298ae407efd6c92c7e (<class 'rdflib.graph.Graph'>)>

## 5) Restricciones OWL y unionOf

In [6]:
exist_restr = BNode()
g.add((exist_restr, RDF.type, OWL.Restriction))
g.add((exist_restr, OWL.onProperty, EX.imponeObligacion))
g.add((exist_restr, OWL.someValuesFrom, EX.ObligacionLaboral))
g.add((EX.LeyLaboral, RDFS.subClassOf, exist_restr))

univ_restr = BNode()
g.add((univ_restr, RDF.type, OWL.Restriction))
g.add((univ_restr, OWL.onProperty, EX.tieneRelacionConActor))
g.add((univ_restr, OWL.allValuesFrom, EX.ActorLaboral))
g.add((EX.ContratoLaboral, RDFS.subClassOf, univ_restr))

card_restr = BNode()
g.add((card_restr, RDF.type, OWL.Restriction))
g.add((card_restr, OWL.onProperty, EX.tieneEmpleador))
g.add((card_restr, OWL.cardinality, Literal(1, datatype=XSD.nonNegativeInteger)))
g.add((EX.ContratoLaboral, RDFS.subClassOf, card_restr))

union_class = BNode()
union_list = BNode()
Collection(g, union_list, [EX.Trabajador, EX.Empleador])
g.add((union_class, RDF.type, OWL.Class))
g.add((union_class, OWL.unionOf, union_list))
g.set((EX.beneficiaA, RDFS.range, union_class))

<Graph identifier=N6af75c0dd1dd49298ae407efd6c92c7e (<class 'rdflib.graph.Graph'>)>

## 6) Instancias y relaciones

In [7]:
def add_individual(uri, class_uri, name):
    g.add((uri, RDF.type, class_uri))
    g.add((uri, EX.tieneNombre, Literal(name)))

for u, n in [(EX.trabajador_ana, "Ana Ruiz"), (EX.trabajador_carlos, "Carlos Mejia"), (EX.trabajador_luisa, "Luisa Gomez"), (EX.trabajador_mario, "Mario Perez")]:
    add_individual(u, EX.Trabajador, n)
for u, n in [(EX.empleador_alfa, "Industria Alfa S.A.S"), (EX.empleador_beta, "Servicios Beta S.A.S"), (EX.empleador_gamma, "Logistica Gamma S.A.S"), (EX.empleador_delta, "Tecnologia Delta S.A.S")]:
    add_individual(u, EX.Empleador, n)
for u, n in [(EX.derecho_salario_justo, "Derecho al salario justo"), (EX.derecho_descanso, "Derecho al descanso"), (EX.derecho_seguridad_social, "Derecho a la seguridad social"), (EX.derecho_licencia_parental, "Derecho a licencias parentales")]:
    add_individual(u, EX.DerechoLaboral, n)
for u, n in [(EX.obligacion_pago_salario, "Pagar salario oportunamente"), (EX.obligacion_afiliacion_seguridad_social, "Afiliar a seguridad social"), (EX.obligacion_prevencion_riesgos, "Prevenir riesgos laborales"), (EX.obligacion_otorgar_descansos, "Otorgar descansos y vacaciones")]:
    add_individual(u, EX.ObligacionLaboral, n)
for u, n in [(EX.sancion_multa_leve, "Multa leve por incumplimiento"), (EX.sancion_multa_grave, "Multa grave por reincidencia"), (EX.sancion_suspension_actividad, "Suspension temporal de actividad"), (EX.sancion_cierre_establecimiento, "Cierre de establecimiento")]:
    add_individual(u, EX.SancionLaboral, n)
for u, n in [(EX.autoridad_mintrabajo, "Ministerio del Trabajo"), (EX.autoridad_inspeccion_medellin, "Inspeccion del Trabajo Medellin"), (EX.autoridad_inspeccion_bogota, "Inspeccion del Trabajo Bogota"), (EX.autoridad_superintendencia_subsidio, "Superintendencia del Subsidio Familiar")]:
    add_individual(u, EX.AutoridadLaboral, n)
for u, n in [(EX.actor_001, "Actor Sindical A"), (EX.actor_002, "Actor Empresarial B"), (EX.actor_003, "Actor Publico C"), (EX.actor_004, "Actor Social D")]:
    add_individual(u, EX.ActorLaboral, n)

for u, n, num, anio in [(EX.norma_general_001, "Codigo Sustantivo del Trabajo", "CST", "1950"), (EX.norma_general_002, "Constitucion Politica de Colombia", "CP", "1991"), (EX.norma_general_003, "Convenio OIT 155", "OIT-155", "1981"), (EX.norma_general_004, "Convenio OIT 187", "OIT-187", "2006")]:
    add_individual(u, EX.NormaJuridica, n)
    g.add((u, EX.tieneNumeroNorma, Literal(num)))
    g.add((u, EX.tieneAnioPublicacion, Literal(anio, datatype=XSD.gYear)))

for u, n, num, anio in [(EX.ley_50_1990, "Ley 50 de 1990", "50", "1990"), (EX.ley_2088_2021, "Ley 2088 de 2021", "2088", "2021"), (EX.ley_2114_2021, "Ley 2114 de 2021", "2114", "2021"), (EX.ley_2209_2022, "Ley 2209 de 2022", "2209", "2022")]:
    add_individual(u, EX.LeyLaboral, n)
    g.add((u, EX.tieneNumeroNorma, Literal(num)))
    g.add((u, EX.tieneAnioPublicacion, Literal(anio, datatype=XSD.gYear)))

contracts = [
    (EX.contrato_001, "Contrato a termino indefinido 001", "3200000.00", "2024-01-15", 48, EX.trabajador_ana, EX.empleador_alfa),
    (EX.contrato_002, "Contrato teletrabajo 002", "2800000.00", "2024-03-01", 42, EX.trabajador_carlos, EX.empleador_beta),
    (EX.contrato_003, "Contrato fijo 003", "2500000.00", "2024-06-01", 46, EX.trabajador_luisa, EX.empleador_gamma),
    (EX.contrato_004, "Contrato aprendizaje 004", "1500000.00", "2024-08-15", 40, EX.trabajador_mario, EX.empleador_delta),
]
for u, n, s, f, h, t, e in contracts:
    add_individual(u, EX.ContratoLaboral, n)
    g.add((u, EX.salarioMensual, Literal(s, datatype=XSD.decimal)))
    g.add((u, EX.fechaInicioContrato, Literal(f, datatype=XSD.date)))
    g.add((u, EX.horasSemanales, Literal(h, datatype=XSD.integer)))
    g.add((u, EX.tieneTrabajador, t))
    g.add((u, EX.tieneEmpleador, e))

g.add((EX.ley_50_1990, EX.otorgaDerecho, EX.derecho_salario_justo))
g.add((EX.ley_50_1990, EX.otorgaDerecho, EX.derecho_descanso))
g.add((EX.ley_2114_2021, EX.otorgaDerecho, EX.derecho_licencia_parental))
g.add((EX.ley_2088_2021, EX.otorgaDerecho, EX.derecho_descanso))
g.add((EX.ley_2209_2022, EX.otorgaDerecho, EX.derecho_seguridad_social))
g.add((EX.ley_50_1990, EX.imponeObligacion, EX.obligacion_pago_salario))
g.add((EX.ley_2088_2021, EX.imponeObligacion, EX.obligacion_prevencion_riesgos))
g.add((EX.ley_2114_2021, EX.imponeObligacion, EX.obligacion_otorgar_descansos))
g.add((EX.ley_2209_2022, EX.imponeObligacion, EX.obligacion_afiliacion_seguridad_social))

<Graph identifier=N6af75c0dd1dd49298ae407efd6c92c7e (<class 'rdflib.graph.Graph'>)>

## 7) Validacion rapida

In [8]:
def count_instances(class_uri):
    return sum(1 for _ in g.triples((None, RDF.type, class_uri)))

print("Triples totales:", len(g))
for cname in classes:
    print(f"{cname}:", count_instances(EX[cname]))

Triples totales: 234
NormaJuridica: 4
LeyLaboral: 4
ActorLaboral: 4
Trabajador: 4
Empleador: 4
ContratoLaboral: 4
DerechoLaboral: 4
ObligacionLaboral: 4
SancionLaboral: 4
AutoridadLaboral: 4


## 8) Exportar a Turtle

In [9]:
output_path = Path("ontologia_practica2.ttl")
g.serialize(destination=str(output_path), format="turtle")
print("Archivo generado:", output_path.resolve())

Archivo generado: C:\Github\Agentic-AI-practice\ontologia_practica2.ttl


## 9) Cargar Ontología en Memoria y Aplicar Razonamiento OWL-RL

In [4]:
from rdflib import Graph
import owlrl

# Crear grafo en memoria y cargar la ontología
print("📥 Cargando ontología en memoria...")
ontology_graph = Graph()
ontology_graph.parse("C:\Github\Agentic-AI-practice\ontologia_practica2.ttl", format="ttl")

print(f"✅ Ontología cargada: {len(ontology_graph)} triples")

# Aplicar razonamiento OWL-RL
print("\n🧠 Aplicando razonamiento OWL-RL...")
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(ontology_graph)

print(f"✅ Razonamiento completado: {len(ontology_graph)} triples (incluye inferencias)")

# Mostrar cuántas inferencias se generaron
inferencias = len(ontology_graph) - 83  # 83 era el número original
print(f"\n✨ Inferencias generadas: {inferencias} nuevos triples")

📥 Cargando ontología en memoria...
✅ Ontología cargada: 234 triples

🧠 Aplicando razonamiento OWL-RL...
✅ Razonamiento completado: 825 triples (incluye inferencias)

✨ Inferencias generadas: 742 nuevos triples


In [8]:
# Mostrar las clases principales
print("\n🏗️ JERARQUÍA DE CLASES:")
print("="*50)
for s, p, o in ontology_graph.triples((None, RDFS.subClassOf, None)):
    subclass_label = ontology_graph.value(s, RDFS.label, default=s.split('#')[-1])
    parent_label = ontology_graph.value(o, RDFS.label, default=o.split('#')[-1])
    print(f"  • {subclass_label} ⊆ {parent_label}")

print("\n🔗 PROPIEDADES DEFINIDAS:")
print("="*50)
for s in ontology_graph.subjects(RDF.type, OWL.ObjectProperty):
    label = ontology_graph.value(s, RDFS.label, default=s.split('#')[-1])
    print(f"  • {label}")


🏗️ JERARQUÍA DE CLASES:
  • Ley Laboral ⊆ n868c7951d83b4370b132d50a8e3333b2b4
  • n868c7951d83b4370b132d50a8e3333b2b4 ⊆ n868c7951d83b4370b132d50a8e3333b2b4
  • Nothing ⊆ n868c7951d83b4370b132d50a8e3333b2b4
  • Ley Laboral ⊆ Norma Juridica
  • Nothing ⊆ Norma Juridica
  • Norma Juridica ⊆ Norma Juridica
  • Autoridad Laboral ⊆ Actor Laboral
  • Empleador ⊆ Actor Laboral
  • Trabajador ⊆ Actor Laboral
  • Nothing ⊆ Actor Laboral
  • Actor Laboral ⊆ Actor Laboral
  • Contrato Laboral ⊆ n868c7951d83b4370b132d50a8e3333b2b5
  • Nothing ⊆ n868c7951d83b4370b132d50a8e3333b2b5
  • Contrato Laboral ⊆ n868c7951d83b4370b132d50a8e3333b2b6
  • Nothing ⊆ n868c7951d83b4370b132d50a8e3333b2b6
  • n868c7951d83b4370b132d50a8e3333b2b6 ⊆ n868c7951d83b4370b132d50a8e3333b2b6
  • Obligacion Laboral ⊆ Obligacion Laboral
  • Nothing ⊆ Obligacion Laboral
  • Thing ⊆ Thing
  • Norma Juridica ⊆ Thing
  • Derecho Laboral ⊆ Thing
  • Sancion Laboral ⊆ Thing
  • Autoridad Laboral ⊆ Thing
  • Ley Laboral ⊆ Thing
  • Co

## 10) Exportar a grafo expandido

In [9]:
ontology_graph.serialize(destination="grafo_extenso.ttl", format="turtle")

<Graph identifier=Nd52c3842d8d94dacb1b7fafc84336237 (<class 'rdflib.graph.Graph'>)>